# frozenlake-rl: every classical RL method on FrozenLake, Taxi and CliffWalking

This notebook goes through the methods of the `frozenlake-rl` repository one by one, in the
order they build on each other, and then runs the full comparison:

| part | methods | needs the model P? | learns from |
|---|---|---|---|
| 1. Dynamic programming | value iteration, policy iteration | yes | nothing, it computes |
| 2. Monte Carlo | first-visit / every-visit MC control | no | complete episodes |
| 3. Temporal difference | SARSA, expected SARSA, Q-learning, double Q-learning | no | single steps |
| 4. Multi-step | n-step SARSA, SARSA(lambda), Q(lambda) | no | several steps / traces |
| 5. Planning | Dyna-Q | learns its own | real + simulated steps |
| 6. Neural | DQN, REINFORCE on one-hot states | no | replay / whole episodes |

Everything runs on a CPU (Colab or Kaggle without GPU is fine). The small problems take
seconds; the full comparison at the end takes 20 to 40 minutes.

In [ ]:
# Install the package (Kaggle: turn Internet on in the settings panel).
# The repo is private: on Colab/Kaggle, add a GitHub token as the secret GITHUB_TOKEN, or
# upload the repo folder and run `pip install -e <folder>` instead.
import os, subprocess, sys
token = os.environ.get("GITHUB_TOKEN", "")
try:
    from google.colab import userdata  # Colab secrets
    token = token or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient  # Kaggle secrets
    token = token or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"git+https://{token + '@' if token else ''}github.com/benisalla/frozenlake-rl"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", url], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from frozenlake_rl.common import set_threads
from frozenlake_rl.config import default_config
from frozenlake_rl.envs import make_env, transition_model
from frozenlake_rl.train import train, optimal_score
from frozenlake_rl.evaluate import ExactEvaluator, rollout_evaluate
from frozenlake_rl.compare import compare
from frozenlake_rl import viz

set_threads(1)
%matplotlib inline

## 0. The problem: a finite Markov decision process

FrozenLake is a grid: start `S`, frozen cells `F`, holes `H`, goal `G`. Reaching the goal gives
reward 1, everything else 0; falling in a hole ends the episode. On *slippery* ice the agent
moves in the intended direction only 1/3 of the time and slides to one of the two
perpendicular directions otherwise.

An MDP is: states S, actions A, and the dynamics p(s', r | s, a). Toy-text environments give
the dynamics explicitly as `env.unwrapped.P[s][a] = [(prob, s', r, terminated), ...]`. Only
dynamic programming reads it; every other method only calls `reset()` and `step()`.

**Why does it matter?** With P you can *compute* the best policy; without it you must *try*
actions and learn from the results, which is the whole difficulty of reinforcement learning.

In [ ]:
env = make_env("frozenlake-4x4-slippery", render_mode="ansi")
env.reset(seed=0)
print(env.render())
P = transition_model(env)
print("from state 14, action RIGHT (2):", P[14][2])   # 3 outcomes, 1/3 each, one reaches the goal

The value of a policy pi is the expected discounted return from each state, and it obeys the
**Bellman equation** (one step of the dynamics, then the value of where you land):

$$v_\pi(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r|s, a)\,[r + \gamma v_\pi(s')]$$

The optimal value obeys the **Bellman optimality equation**, with a max instead of the average
over actions:

$$v_*(s) = \max_a \sum_{s', r} p(s', r|s, a)\,[r + \gamma v_*(s')], \qquad
q_*(s, a) = \sum_{s', r} p(s', r|s, a)\,[r + \gamma \max_{a'} q_*(s', a')]$$

Every method below is a different way of solving (or approximately solving) one of these
equations: DP iterates them with the true p, MC replaces the expectation by full sampled
returns, TD replaces it by one sampled step plus the current estimate.

## 1. Dynamic programming: value iteration and policy iteration

**Value iteration** turns the optimality equation into an update and applies it to every state
until nothing changes: `V(s) <- max_a sum p [r + gamma V(s')]`. Each sweep propagates the
goal's reward one step further back.

**Policy iteration** alternates two steps: *evaluate* the current policy completely (solve the
Bellman equation for v_pi), then *improve* it greedily. It needs very few improvement steps
(about 7 here, against hundreds of value-iteration sweeps) because each one is a full, exact evaluation.

Why does it work? The Bellman backup is a gamma-contraction: every sweep shrinks the error by
at least a factor gamma, so it converges to the unique fixed point v_*.

The package code (`src/frozenlake_rl/algorithms/dp.py`):

```python
def value_iteration(env, cfg, rng, tracker) -> dict:
    T, R = model_arrays(env)
    V = np.zeros(T.shape[0])
    tracker.start(q_from_v(T, R, V, cfg.gamma).argmax(1))
    deltas = []
    for sweep in range(1, cfg.max_sweeps + 1):
        Q = q_from_v(T, R, V, cfg.gamma)
        V_new = Q.max(1)                          # Bellman optimality backup, all states at once
        delta = float(np.abs(V_new - V).max())
        V = V_new
        deltas.append(delta)
        tracker.end_sweep(sweep, Q.argmax(1))
        if delta < cfg.theta:
            break
    Q = q_from_v(T, R, V, cfg.gamma)
    return {"Q": Q, "V": V, "sweeps": len(deltas), "deltas": np.array(deltas)}

def policy_iteration(env, cfg, rng, tracker) -> dict:
    T, R = model_arrays(env)
    S = T.shape[0]
    policy = np.zeros(S, dtype=np.int64)
    V = np.zeros(S)
    tracker.start(policy)
    eval_sweeps = []
    for step in range(1, cfg.max_sweeps + 1):
        V, n = policy_evaluation(T, R, policy, cfg.gamma, cfg.theta, cfg.max_sweeps, V)   # evaluation
        eval_sweeps.append(n)
        Q = q_from_v(T, R, V, cfg.gamma)
        # improvement; keep the old action on (numerical) ties so the loop terminates
        best = Q.max(1)
        keep = Q[np.arange(S), policy] >= best - 1e-12
        new_policy = np.where(keep, policy, Q.argmax(1))
        tracker.end_sweep(step, new_policy)
        if np.array_equal(new_policy, policy):
            break
        policy = new_policy
    return {"Q": Q, "V": V, "policy": policy, "improvements": len(eval_sweeps),
            "sweeps": int(sum(eval_sweeps)), "eval_sweeps": np.array(eval_sweeps)}
```

In [ ]:
vi = train(default_config("frozenlake-4x4-slippery", "value-iteration"))
pi = train(default_config("frozenlake-4x4-slippery", "policy-iteration"))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
viz.plot_value_policy(vi.V, vi.policy, "frozenlake-4x4-slippery", axes[0], "value iteration")
viz.plot_value_policy(pi.V, pi.policy, "frozenlake-4x4-slippery", axes[1], "policy iteration")
plt.show()
print("same values:", np.allclose(vi.V, pi.V, atol=1e-6), "| sweeps VI:", vi.summary["sweeps"],
      "| improvements PI:", pi.extras["improvements"])

Look at the arrows: on slippery ice the optimal policy often points *away* from the hole
(left in the first column, up on the top row), because moving "left" can only slip up or
down, never into the hole on the right. Even the best policy only succeeds about 74 % of the
time: that number is the ceiling every learning method below is measured against.

### How every method is scored

The same two measurements for every method:

* **exact**: the success probability (or expected return) of the greedy policy computed from
  P with the 100/200-step time limit. No sampling noise, so the learning curves are smooth.
  Only the experimenter uses P here; the agents never see it.
* **rollouts**: N real gymnasium episodes with the greedy policy, the number you would get
  without knowing the model. Both agree within the Monte Carlo error.

The package code (`src/frozenlake_rl/evaluate.py`):

```python
class ExactEvaluator:
    """Finite-horizon policy evaluation on the true model.

    For a horizon of H steps (the time limit) and policy pi:

        v_0 = 0,    v_k(s) = sum_a pi(a|s) sum_{s'} p(s'|s,a) [ r + (1 - done) v_{k-1}(s') ]

    v_H at the start distribution is the expected return of one episode; replacing r by the
    indicator "this transition is a success" gives the success probability.
    """

    def __init__(self, env_name: str):
        self.spec = get_spec(env_name)
        env = make_env(env_name)
        P = transition_model(env)
        S, A = n_states_actions(env)
        self.n_states, self.n_actions = S, A
        self.R = np.zeros((S, A))          # expected immediate reward
        self.SU = np.zeros((S, A))         # probability that this step is a success
        self.M = np.zeros((S, A, S))       # probability of continuing to s'
        for s in range(S):
            for a in range(A):
                for p, s2, r, done in P[s][a]:
                    self.R[s, a] += p * r
                    self.SU[s, a] += p * is_success(self.spec, done, r)
                    if not done:
                        self.M[s, a, s2] += p
        self.start = np.asarray(env.unwrapped.initial_state_distrib, dtype=np.float64)
        env.close()

    def __call__(self, policy: np.ndarray) -> dict:
        pi = policy_matrix(policy, self.n_actions)
        r_pi = (pi * self.R).sum(1)
        s_pi = (pi * self.SU).sum(1)
        m_pi = np.einsum("sa,sat->st", pi, self.M)
        v_ret = np.zeros(self.n_states)
        v_suc = np.zeros(self.n_states)
        for _ in range(self.spec.max_steps):
            v_ret = r_pi + m_pi @ v_ret
            v_suc = s_pi + m_pi @ v_suc
        return {"success": float(self.start @ v_suc), "return": float(self.start @ v_ret)}


    def best(self) -> dict:
        """The best score any policy can reach within the time limit (finite-horizon DP).

        Maximises the env's target metric directly: v_k(s) = max_a [ score(s, a) + sum_s' M v_{k-1}(s') ]
        with score = success indicator (FrozenLake) or reward (Taxi, CliffWalking). This is the
        ceiling of the evaluation metric; the gamma-discounted optimum of value iteration can sit
        slightly below it (gamma = 0.99 trades a little success for speed on the 8x8 lake).
        """
        score = self.SU if self.spec.target_metric == "success" else self.R
        v = np.zeros(self.n_states)
        for _ in range(self.spec.max_steps):
            v = (score + self.M @ v).max(1)
        return {self.spec.target_metric: float(self.start @ v)}
```

In [ ]:
print("exact   :", ExactEvaluator("frozenlake-4x4-slippery")(vi.policy))
print("rollouts:", rollout_evaluate("frozenlake-4x4-slippery", vi.policy, episodes=2000))
print("uniform random policy:", ExactEvaluator("frozenlake-4x4-slippery")(np.full((16, 4), 0.25)))

## 2. Monte Carlo control: learn from complete episodes

No model now. Play a whole episode with an epsilon-greedy policy, compute the actual return
G_t that followed each (s, a), and average it into Q(s, a). Then act greedily w.r.t. the new
Q (with epsilon exploration). This is *generalised policy iteration* with sampled evaluation.

* **first-visit** averages only the first occurrence of (s, a) per episode (unbiased);
  **every-visit** averages all of them (more updates, slightly biased, converges too).
* **exploring starts** begin each episode in a random state with a random action, so every
  pair is tried even if the policy never goes there.

Why is it slow on big problems? G_t sums many random rewards, so its variance grows with the
episode length, and on FrozenLake most episodes end in a hole with G = 0: nothing is learned
until the goal is hit by chance.

The package code (`src/frozenlake_rl/algorithms/monte_carlo.py`):

```python
def mc_control(env, cfg, rng, tracker, first_visit: bool = True) -> dict:
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q = np.zeros((S, A))
    N = np.zeros((S, A))
    sched = Schedules(cfg)
    starts = np.flatnonzero(~terminal_states(env))
    tracker.start(greedy_policy(Q))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        s, _ = env.reset(seed=cfg.seed if episode == 0 else None)
        first_action = None
        if cfg.exploring_starts:
            s = int(rng.choice(starts))
            env.unwrapped.s = s                     # teleport: legal for toy-text envs
            first_action = int(rng.integers(A))

        # 1. generate an episode with the current epsilon-greedy policy
        trajectory = []                             # (s, a, r)
        terminated = truncated = False
        while not (terminated or truncated):
            a = first_action if (first_action is not None and not trajectory) else epsilon_greedy(Q[s], eps, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            trajectory.append((int(s), int(a), float(r)))
            s = s2

        # 2. walk it backwards and average the returns into Q
        if first_visit:
            first_index = {}
            for t, (st, at, _r) in enumerate(trajectory):
                first_index.setdefault((st, at), t)
        G = 0.0
        for t in range(len(trajectory) - 1, -1, -1):
            st, at, r = trajectory[t]
            G = r + cfg.gamma * G
            if first_visit and first_index[(st, at)] != t:
                continue
            N[st, at] += 1
            step = 1.0 / N[st, at] if cfg.sample_average else alpha
            Q[st, at] += step * (G - Q[st, at])

        total = sum(r for _s, _a, r in trajectory)
        tracker.end_episode(episode + 1, total, len(trajectory), is_success(spec, terminated, total),
                            lambda: greedy_policy(Q))
    return {"Q": Q, "V": Q.max(1), "visits": N}
```

In [ ]:
mc = train(default_config("frozenlake-4x4-slippery", "mc-first-visit"))
mc_es = train(default_config("frozenlake-4x4-slippery", "mc-first-visit", exploring_starts=True))
viz.plot_curves({"mc first-visit": mc.history, "mc + exploring starts": mc_es.history},
                "frozenlake-4x4-slippery", optimum=vi.summary["optimum"]); plt.show()

## 3. Temporal-difference learning: learn from every single step

TD methods update after each step towards a target that uses the *current estimate* of the
next state (bootstrapping): `Q(s,a) <- Q(s,a) + alpha [target - Q(s,a)]`.

| method | target | on/off policy |
|---|---|---|
| SARSA | r + gamma Q(s', a') with the a' actually taken next | on-policy: learns the value of the exploring policy |
| expected SARSA | r + gamma sum_a' pi(a'|s') Q(s', a') | on-policy, no sampling noise from a' |
| Q-learning | r + gamma max_a' Q(s', a') | off-policy: learns the greedy policy while exploring |
| double Q-learning | r + gamma Q2(s', argmax Q1(s', .)) | off-policy, removes the max over noisy estimates bias |

**On-policy vs off-policy** in one picture: CliffWalking. With epsilon = 0.1 fixed, Q-learning
learns the optimal path right along the cliff, but its exploring agent keeps falling in
(-100). SARSA accounts for its own exploration and learns the safer path one row up: worse
greedy policy, better returns while learning (Sutton & Barto, example 6.6).

The package code (`src/frozenlake_rl/algorithms/td.py`):

```python
def _one_step_td(env, cfg, rng, tracker, target_fn) -> dict:
    """The loop shared by SARSA, expected SARSA and Q-learning; only `target_fn` differs.

    target_fn(Q, s2, a2, epsilon) returns the value of the next state used in the target.
    """
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q = np.zeros((S, A))
    sched = Schedules(cfg)
    tracker.start(greedy_policy(Q))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        s = _reset(env, cfg, episode)
        a = epsilon_greedy(Q[s], eps, rng)
        total, steps = 0.0, 0
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = epsilon_greedy(Q[s2], eps, rng)
            future = 0.0 if terminated else target_fn(Q, s2, a2, eps)
            Q[s, a] += alpha * (r + cfg.gamma * future - Q[s, a])
            total += r
            steps += 1
            if terminated or truncated:
                break
            s, a = s2, a2
        tracker.end_episode(episode + 1, total, steps, is_success(spec, terminated, total),
                            lambda: greedy_policy(Q))
    return {"Q": Q, "V": Q.max(1)}

def sarsa(env, cfg, rng, tracker) -> dict:
    """On-policy TD: bootstrap on the action a' the agent will actually take next."""
    return _one_step_td(env, cfg, rng, tracker, lambda Q, s2, a2, eps: Q[s2, a2])

def expected_sarsa(env, cfg, rng, tracker) -> dict:
    """SARSA with the sampled a' replaced by its expectation under epsilon-greedy: less variance."""
    return _one_step_td(env, cfg, rng, tracker,
                        lambda Q, s2, a2, eps: float(epsilon_greedy_probs(Q[s2], eps) @ Q[s2]))

def q_learning(env, cfg, rng, tracker) -> dict:
    """Off-policy TD: bootstrap on the best next action, whatever the agent does next."""
    return _one_step_td(env, cfg, rng, tracker, lambda Q, s2, a2, eps: Q[s2].max())
```

In [ ]:
cliff = {m: train(default_config("cliffwalking", m)) for m in ["sarsa", "expected-sarsa", "q-learning"]}
viz.plot_train_returns({m: r.train_returns for m, r in cliff.items()}, "cliffwalking", window=20, ylim=(-150, 0))
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 2.4))
viz.plot_value_policy(cliff["sarsa"].V, cliff["sarsa"].policy, "cliffwalking", axes[0], "SARSA")
viz.plot_value_policy(cliff["q-learning"].V, cliff["q-learning"].policy, "cliffwalking", axes[1], "Q-learning")
plt.show()

The package code (`src/frozenlake_rl/algorithms/td.py`):

```python
def double_q_learning(env, cfg, rng, tracker) -> dict:
    """Two tables: one picks the argmax, the other evaluates it (van Hasselt 2010).

    max_a Q(s', a) over noisy estimates is biased upwards; decoupling selection from
    evaluation removes that maximisation bias. Behaviour is epsilon-greedy on Q1 + Q2.
    """
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q1, Q2 = np.zeros((S, A)), np.zeros((S, A))
    sched = Schedules(cfg)
    tracker.start(greedy_policy(Q1 + Q2))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        s = _reset(env, cfg, episode)
        total, steps = 0.0, 0
        while True:
            a = epsilon_greedy(Q1[s] + Q2[s], eps, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            if rng.random() < 0.5:
                update, other = Q1, Q2
            else:
                update, other = Q2, Q1
            future = 0.0 if terminated else other[s2, int(update[s2].argmax())]
            update[s, a] += alpha * (r + cfg.gamma * future - update[s, a])
            total += r
            steps += 1
            if terminated or truncated:
                break
            s = s2
        tracker.end_episode(episode + 1, total, steps, is_success(spec, terminated, total),
                            lambda: greedy_policy(Q1 + Q2))
    Q = (Q1 + Q2) / 2
    return {"Q": Q, "V": Q.max(1), "Q1": Q1, "Q2": Q2}
```

In [ ]:
td = {m: train(default_config("frozenlake-4x4-slippery", m))
      for m in ["sarsa", "expected-sarsa", "q-learning", "double-q-learning"]}
viz.plot_curves({m: r.history for m, r in td.items()}, "frozenlake-4x4-slippery",
                optimum=vi.summary["optimum"]); plt.show()

## 4. Between MC and TD: n-step returns and eligibility traces

One-step TD propagates a reward back by one state per visit; MC propagates it along the whole
episode but with high variance. **n-step SARSA** uses n real rewards then bootstraps:

$$G_{t:t+n} = r_{t+1} + \gamma r_{t+2} + \dots + \gamma^{n-1} r_{t+n} + \gamma^n Q(s_{t+n}, a_{t+n})$$

**Eligibility traces** do the same online and for all n at once (the lambda-return): each
visited pair keeps a trace E(s, a) that fades by gamma*lambda per step, and every TD error
updates all pairs in proportion to their trace. One reward at the goal immediately updates
the whole path that led to it. **Q(lambda)** (Watkins) is the off-policy version: it cuts the
traces after an exploratory action, because what followed is not the greedy path.

The package code (`src/frozenlake_rl/algorithms/td.py`):

```python
def n_step_sarsa(env, cfg, rng, tracker) -> dict:
    """n-step SARSA (Sutton & Barto, section 7.2): n real rewards, then bootstrap.

    n = 1 is SARSA; n = infinity is Monte Carlo. The update of time tau happens at time
    tau + n, once the n rewards after it are known.
    """
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q = np.zeros((S, A))
    n, gamma = cfg.n, cfg.gamma
    sched = Schedules(cfg)
    tracker.start(greedy_policy(Q))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        states = [_reset(env, cfg, episode)]
        actions = [epsilon_greedy(Q[states[0]], eps, rng)]
        rewards = [0.0]                                   # rewards[t] is R_t (R_0 unused)
        T, t, bootstrap_end, terminated = np.inf, 0, False, False
        while True:
            if t < T:
                s2, r, terminated, truncated, _ = env.step(actions[t])
                states.append(int(s2))
                rewards.append(float(r))
                if terminated or truncated:
                    T = t + 1
                    bootstrap_end = not terminated          # truncated: S_T still has a future
                actions.append(epsilon_greedy(Q[s2], eps, rng))
            tau = t - n + 1                                 # the time whose estimate is updated
            if tau >= 0:
                end = int(min(tau + n, T))
                G = sum(gamma ** (i - tau - 1) * rewards[i] for i in range(tau + 1, end + 1))
                if tau + n < T or bootstrap_end:
                    G += gamma ** (end - tau) * Q[states[end], actions[end]]
                sa = (states[tau], actions[tau])
                Q[sa] += alpha * (G - Q[sa])
            if tau == T - 1:
                break
            t += 1
        total = float(sum(rewards))
        tracker.end_episode(episode + 1, total, int(T), is_success(spec, terminated, total),
                            lambda: greedy_policy(Q))
    return {"Q": Q, "V": Q.max(1)}

def _lambda_control(env, cfg, rng, tracker, watkins: bool) -> dict:
    """SARSA(lambda) or Watkins's Q(lambda) with replacing eligibility traces.

    The trace E(s, a) remembers how recently (s, a) was visited; every TD error delta updates
    *all* pairs at once, Q += alpha * delta * E, then traces fade: E *= gamma * lambda. A single
    reward thus reaches every state of the path in one episode, like Monte Carlo, while still
    learning online, like TD. Q(lambda) bootstraps on max Q and cuts the traces when the agent
    takes an exploratory (non-greedy) action, because the path after it is not the greedy one.
    """
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q = np.zeros((S, A))
    E = np.zeros((S, A))
    decay = cfg.gamma * cfg.lam
    sched = Schedules(cfg)
    tracker.start(greedy_policy(Q))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        E[:] = 0.0
        s = _reset(env, cfg, episode)
        a = epsilon_greedy(Q[s], eps, rng)
        total, steps = 0.0, 0
        while True:
            s2, r, terminated, truncated, _ = env.step(a)
            a2 = epsilon_greedy(Q[s2], eps, rng)
            next_is_greedy = Q[s2, a2] >= Q[s2].max()
            if terminated:
                future = 0.0
            else:
                future = Q[s2].max() if watkins else Q[s2, a2]
            delta = r + cfg.gamma * future - Q[s, a]
            E[s, :] = 0.0                       # replacing trace (and clear the other actions of s)
            E[s, a] = 1.0
            Q += alpha * delta * E
            if watkins and not next_is_greedy:
                E[:] = 0.0                      # exploratory action: the greedy path is broken
            else:
                E *= decay
            total += r
            steps += 1
            if terminated or truncated:
                break
            s, a = s2, a2
        tracker.end_episode(episode + 1, total, steps, is_success(spec, terminated, total),
                            lambda: greedy_policy(Q))
    return {"Q": Q, "V": Q.max(1)}
```

In [ ]:
multi = {m: train(default_config("frozenlake-8x8", m)) for m in ["sarsa", "n-step-sarsa", "sarsa-lambda", "q-lambda"]}
viz.plot_curves({m: r.history for m, r in multi.items()}, "frozenlake-8x8"); plt.show()

## 5. Dyna-Q: learn a model, then plan with it

Dyna-Q is Q-learning plus a learned model. After every real step it stores what happened
(counts of (s', r) per (s, a), so slippery outcomes keep their probabilities) and then performs
`planning_steps` extra Q-learning updates on transitions *replayed from that model*. Each real
experience is used many times, so far fewer real episodes are needed: it sits between DP
(full model, no experience) and Q-learning (no model, only experience).

The package code (`src/frozenlake_rl/algorithms/planning.py`):

```python
class CountModel:
    """Empirical model: for each visited (s, a), how often each (s', r, terminated) followed."""

    def __init__(self):
        self.outcomes: dict[tuple[int, int], dict[tuple[int, float, bool], int]] = {}
        self.visited: list[tuple[int, int]] = []

    def update(self, s: int, a: int, s2: int, r: float, terminated: bool) -> None:
        key = (s, a)
        if key not in self.outcomes:
            self.outcomes[key] = {}
            self.visited.append(key)
        table = self.outcomes[key]
        out = (s2, r, terminated)
        table[out] = table.get(out, 0) + 1

    def sample(self, rng: np.random.Generator) -> tuple[int, int, int, float, bool]:
        s, a = self.visited[int(rng.integers(len(self.visited)))]
        table = self.outcomes[(s, a)]
        outcomes = list(table)
        cumulative = np.cumsum(list(table.values()))     # sample an outcome with probability count / total
        i = int(np.searchsorted(cumulative, rng.random() * cumulative[-1], side="right"))
        s2, r, terminated = outcomes[i]
        return s, a, s2, r, terminated

def dyna_q(env, cfg, rng, tracker) -> dict:
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    Q = np.zeros((S, A))
    model = CountModel()
    sched = Schedules(cfg)
    gamma = cfg.gamma
    tracker.start(greedy_policy(Q))
    for episode in range(cfg.episodes):
        eps, alpha = sched.epsilon(episode), sched.alpha(episode)
        s, _ = env.reset(seed=cfg.seed if episode == 0 else None)
        total, steps = 0.0, 0
        while True:
            a = epsilon_greedy(Q[s], eps, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            # (1) direct RL: Q-learning on the real transition
            Q[s, a] += alpha * (r + (0.0 if terminated else gamma * Q[s2].max()) - Q[s, a])
            # (2) model learning
            model.update(int(s), int(a), int(s2), float(r), bool(terminated))
            # (3) planning: replay simulated transitions from the model
            for _ in range(cfg.planning_steps):
                ps, pa, ps2, pr, pterm = model.sample(rng)
                Q[ps, pa] += alpha * (pr + (0.0 if pterm else gamma * Q[ps2].max()) - Q[ps, pa])
            total += r
            steps += 1
            if terminated or truncated:
                break
            s = s2
        tracker.end_episode(episode + 1, total, steps, is_success(spec, terminated, total),
                            lambda: greedy_policy(Q))
    return {"Q": Q, "V": Q.max(1)}
```

In [ ]:
plan = {f"dyna-q (n={k})": train(default_config("frozenlake-8x8", "dyna-q", planning_steps=k)) for k in [0, 5, 20]}
viz.plot_curves({m: r.history for m, r in plan.items()}, "frozenlake-8x8"); plt.show()

## 6. Neural networks on one-hot states: DQN and REINFORCE

The deep-RL machinery on a problem where the answer is known. A state becomes a one-hot vector,
so the first linear layer is literally a table; the network adds nothing a table cannot do,
which makes it a clean test of the training loops.

* **DQN**: Q-learning with an MLP, a replay buffer (break correlations between consecutive
  samples) and a target network (a frozen copy for the target, so the regression target does
  not move at every step).
* **REINFORCE**: learn the policy directly, `grad = E[grad log pi(a|s) (G_t - V(s))]`, with a
  learned value baseline to cut the variance and an entropy bonus to keep exploring.

The package code (`src/frozenlake_rl/algorithms/neural.py`):

```python
def dqn(env, cfg, rng, tracker) -> dict:
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    torch.manual_seed(cfg.seed)
    eye = torch.eye(S)                               # row s is the one-hot encoding of state s
    online, target = mlp(S, cfg.hidden, A), mlp(S, cfg.hidden, A)
    target.load_state_dict(online.state_dict())
    optimizer = torch.optim.Adam(online.parameters(), lr=cfg.lr)
    buffer = ReplayBuffer(cfg.buffer_size)
    sched = Schedules(cfg)

    @torch.no_grad()
    def q_table() -> np.ndarray:
        return online(eye).numpy().astype(np.float64)

    tracker.start(q_table().argmax(1))
    q_cache = q_table()                              # Q of all states, refreshed after each update
    total_steps = updates = 0
    losses = []
    for episode in range(cfg.episodes):
        eps = sched.epsilon(episode)
        s, _ = env.reset(seed=cfg.seed if episode == 0 else None)
        total, steps = 0.0, 0
        while True:
            a = epsilon_greedy(q_cache[s], eps, rng)
            s2, r, terminated, truncated, _ = env.step(a)
            buffer.add(s, a, r, s2, float(terminated))
            total += r
            steps += 1
            total_steps += 1
            if buffer.size >= cfg.learning_starts and total_steps % cfg.train_every == 0:
                bs, ba, br, bs2, bd = buffer.sample(cfg.batch_size, rng)
                with torch.no_grad():
                    y = torch.from_numpy(br) + cfg.gamma * (1 - torch.from_numpy(bd)) * target(eye[bs2]).max(1).values
                q = online(eye[bs]).gather(1, torch.from_numpy(ba)[:, None]).squeeze(1)
                loss = nn.functional.smooth_l1_loss(q, y)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(online.parameters(), 10.0)
                optimizer.step()
                updates += 1
                losses.append(loss.item())
                if updates % cfg.target_update == 0:
                    target.load_state_dict(online.state_dict())
                q_cache = q_table()
            if terminated or truncated:
                break
            s = s2
        tracker.end_episode(episode + 1, total, steps, is_success(spec, terminated, total),
                            lambda: q_cache.argmax(1))
    Q = q_table()
    return {"Q": Q, "V": Q.max(1), "losses": np.array(losses), "model": online.state_dict()}

def reinforce(env, cfg, rng, tracker) -> dict:
    spec = get_spec(cfg.env)
    S, A = n_states_actions(env)
    torch.manual_seed(cfg.seed)
    eye = torch.eye(S)
    policy_net, value_net = mlp(S, cfg.hidden, A), mlp(S, cfg.hidden, 1)
    optimizer = torch.optim.Adam(list(policy_net.parameters()) + list(value_net.parameters()), lr=cfg.lr)

    @torch.no_grad()
    def probs_table() -> np.ndarray:
        return torch.softmax(policy_net(eye), -1).numpy().astype(np.float64)

    probs = probs_table()
    tracker.start(probs.argmax(1))
    batch: list[tuple[list, list, np.ndarray]] = []          # episodes waiting for the next update
    for episode in range(cfg.episodes):
        s, _ = env.reset(seed=cfg.seed if episode == 0 else None)
        states, actions, rewards = [], [], []
        terminated = truncated = False
        while not (terminated or truncated):
            p = probs[s]
            a = int(rng.choice(A, p=p / p.sum()))        # sample from pi(.|s): exploration is built in
            s2, r, terminated, truncated, _ = env.step(a)
            states.append(int(s))
            actions.append(a)
            rewards.append(float(r))
            s = s2
        # returns-to-go G_t, computed backwards
        G, returns = 0.0, np.zeros(len(rewards), dtype=np.float32)
        for t in range(len(rewards) - 1, -1, -1):
            G = rewards[t] + cfg.gamma * G
            returns[t] = G
        batch.append((states, actions, returns))
        if len(batch) == cfg.episodes_per_update or episode == cfg.episodes - 1:
            x = eye[np.concatenate([b[0] for b in batch])]
            a_t = torch.tensor(np.concatenate([b[1] for b in batch]))[:, None]
            G_t = torch.from_numpy(np.concatenate([b[2] for b in batch]))
            log_pi = torch.log_softmax(policy_net(x), -1)
            values = value_net(x).squeeze(1)
            advantage = (G_t - values).detach()               # baseline: what the critic expected
            policy_loss = -(log_pi.gather(1, a_t).squeeze(1) * advantage).mean()
            entropy = -(log_pi.exp() * log_pi).sum(1).mean()
            value_loss = ((G_t - values) ** 2).mean()
            loss = policy_loss + 0.5 * value_loss - cfg.entropy_coef * entropy
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            probs = probs_table()
            batch = []
        total = float(sum(rewards))
        tracker.end_episode(episode + 1, total, len(rewards), is_success(spec, terminated, total),
                            lambda: probs.argmax(1))
    with torch.no_grad():
        V = value_net(eye).squeeze(1).numpy().astype(np.float64)
    # no Q for a policy-gradient method: store log pi as the "Q" so heatmaps / argmax still work
    return {"Q": np.log(probs + 1e-12), "V": V, "probs": probs, "model": policy_net.state_dict()}
```

In [ ]:
neural = {m: train(default_config("frozenlake-4x4-slippery", m)) for m in ["dqn", "reinforce"]}
viz.plot_curves({m: r.history for m, r in neural.items()}, "frozenlake-4x4-slippery",
                optimum=vi.summary["optimum"]); plt.show()

## 7. The full comparison

Every method with the same evaluation on every environment. `EPISODES_SCALE` multiplies each
method's default episode budget (1 = the short CPU runs of the README; 2 or more gives the slow
learners, Monte Carlo and REINFORCE on the 8x8 slippery lake, the time they need). Taxi and
CliffWalking run DP (the reference), the TD family and Dyna-Q.

In [ ]:
EPISODES_SCALE = 2          # 1 = the budgets of the README tables
SUITES = {
    "frozenlake-4x4": "all",
    "frozenlake-4x4-slippery": "all",
    "frozenlake-8x8": "all",
    "frozenlake-8x8-slippery": "all",
    "taxi": "td",
    "cliffwalking": "td",
}
from frozenlake_rl.compare import parse_methods
from IPython.display import Markdown, display, Image

for env_name, group in SUITES.items():
    compare(env_name, parse_methods(group), f"runs/compare/{env_name}", verbose=False, episodes_scale=EPISODES_SCALE)
    display(Markdown(open(f"runs/compare/{env_name}/results.md").read()))

In [ ]:
for env_name in SUITES:
    for fig_name in ["curves.png", "policies.png"]:
        path = f"runs/compare/{env_name}/{fig_name}"
        if os.path.exists(path):
            display(Image(path))

## What to take away

* **DP** gives the exact optimum in milliseconds, but only because it reads P.
* **Monte Carlo** works on short episodic problems; its variance makes it the slowest learner on
  long or sparse ones (8x8 slippery).
* **TD** learns from every step; **Q-learning** finds the optimal greedy policy, **SARSA**
  the best policy *given that it keeps exploring* (the cliff).
* **n-step / lambda** methods spread sparse rewards faster along the path: fewer episodes.
* **Dyna-Q** reuses experience through a learned model: the fewest real episodes of all.
* **DQN / REINFORCE** reach the same policies with much more compute: the network brings
  nothing on a table, but the training loops are exactly those of the Atari-scale versions.